# 检索后处理与回答约束

这一节处理四种情况：正确片段排得太后、上下文太长、回答没有来源，以及资料不足却给出过强结论。这里的“生成阶段”指候选资料的后处理和回答约束；回答文字是为了演示检查方法而写的固定示例，不是模型生成结果。真正接入回答模型时，也必须保留同样的原文核对和资料不足边界。


## 本页导航

1. **教学排序**：候选中已有正确片段，但排得太后时重排。
2. **压缩**：候选过长时，只保留问题所需的原文。
3. **标注来源**：把回答拆成结论，逐条核对实际证据。
4. **资料不足就拒答**：没有支持时保留边界，不把相似页面当答案。

下方四个教学单元（教学排序、压缩、标注来源、资料不足）的保存输出只来自固定示例与本地规则；它们没有调用回答模型，也不声称真实回答模型效果。Notebook 末尾另有标为“可选真实实验”的 Cross-Encoder 单元：它会在本地加载 BAAI/bge-reranker-large，对同一批候选实际打分并保存真实排序、分数和范围；这部分不应与前面的固定示例混为一谈。

## 找到资料后还要做什么

第一次检索只负责找到一批候选资料。接下来可以删除无关句子或整段不相关的资料，也可以根据完整问题重新排序，再把整理后的文字交给回答模型。

![后处理在 RAG 流程中的位置](./postprocess.png)

整理候选资料可以减少模型要阅读的文字，把更相关的内容放在前面，并降低无关内容对回答的干扰。例如手机问题可能同时找到规格表、评测和评论，可以先保留与问题有关的要点，再把用户真正关心的续航资料放到前面。这一步不能凭空产生资料，也不能弥补第一次检索完全漏掉的内容；因此，本节每个例子都会先确认候选中确实有回答所需的原文。

In [1]:
import sys
from pathlib import Path


def find_course_root(start):
    for folder in (start, *start.parents):
        if (folder / "data" / "dataset/manifest.json").is_file():
            return folder
    raise FileNotFoundError("没有找到教程数据目录，请从本节所在目录运行。")


course_root = find_course_root(Path.cwd())
if str(course_root) not in sys.path:
    sys.path.insert(0, str(course_root))

In [2]:
from common.eval_utils import emit_tutorial_audit

import json
import re

import common.eval_utils as eu
from common.eval_utils import (
    build_bm25_chunk_search,
    build_bm25_search,
    load_query_catalog,
    load_default_chunks,
    load_pdf_pages,
)
from common.nontraining_utils import load_annotation, load_query_controls

CONTROL_CHARS = re.compile(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]')

def clean_text(text):
    return eu.normalize_text(CONTROL_CHARS.sub(' ', str(text)))

def compact_text(text):
    return clean_text(text).replace(' ', '')

def preview(text, limit=180):
    value = clean_text(text)
    return value[:limit] + ('...' if len(value) > limit else '')

def answer_point_coverage(text, points):
    compact = compact_text(text)
    checks = [all(compact_text(part) in compact for part in point) for point in points]
    return sum(checks) / len(checks), checks

def result_pages(results):
    pages = []
    for item in results:
        values = getattr(item, 'pages', None)
        pages.extend(int(page) for page in (values if values is not None else [item.page]))
    return pages

def standard_metrics(results, expected_pages):
    pages = result_pages(results)
    expected = set(int(page) for page in expected_pages)
    found = {page for page in pages if page in expected}
    first = next((rank for rank, page in enumerate(pages, 1) if page in expected), None)
    return {'pages': pages, 'first_required_rank': first, 'required_page_coverage': len(found) / len(expected) if expected else 0.0}

def emit_standard(method, role, case_id, before, after, expected_pages, check_purpose=None, before_override=None, after_override=None, comparison=None):
    payload = {'case_id': case_id, 'method': method, 'role': role,
               'before': before_override or standard_metrics(before, expected_pages),
               'after': after_override or standard_metrics(after, expected_pages)}
    if check_purpose:
        payload['check_purpose'] = check_purpose
    if comparison is not None:
        payload['comparison'] = comparison
    emit_tutorial_audit(payload)

cases = {item["id"]: item for item in load_query_catalog()}
pages = load_pdf_pages()

## 教学排序：把候选资料按完整问题重新排队

从问题集中选出一个真实问题：软间隔支持向量机对拉格朗日乘子 αi 有什么约束？用完整问题做第一阶段检索后，包含约束公式的资料已经在 6 条候选中，但排在第 5 条。这里的重排只改变这 6 条的先后顺序：候选片段、数量和字符数必须完全相同。排序分数只看完整问题和候选正文，预期页码、预期关键词与参考答案仅用于事后检查。

### 为什么需要 Reranking

向量检索使用双编码器（Bi-Encoder）分别编码查询和文档，速度快、可以预先建立索引，但固定维度的向量会丢失细节，查询和文档也没有在同一次注意力计算中直接交互。因此它可能把只共享关键词、却不能回答问题的片段排得很高。

工业系统通常用两阶段漏斗：第一阶段用向量检索或 BM25 从海量语料召回较大的候选集，优先保证召回率；第二阶段用交叉编码器（Cross-Encoder）把完整问题和每个候选拼接后一起输入模型，直接估计相关性，再留下前几条给回答模型。交叉编码器能捕捉否定、因果和词序等细节，准确率通常更高，但不能像双编码器一样预先编码所有文档，检索时需要逐候选计算，所以只适合较小的候选集。

![两阶段检索与交叉编码器概念图（非本页实测）](./reranker2.png)

模型选择要同时看精度、延迟、语言、部署方式和数据安全。可以用 MTEB/CMTEB 等公共基准做初筛，再用自己的问题集检查；常见本地方案有 BAAI/bge-reranker-base/large，商业服务有 Cohere Rerank 和 Jina Reranker。榜单只说明公开基准上的相对表现，不保证在当前《南瓜书》问题上同样有效。

![排序效果概念示意（非本页实测）](./reranker_perform.png)

下面先展示接入示意；本页末尾的“可选真实实验”会用同一 canonical query、同一 6 条候选和 top-k 实际运行 BAAI/bge-reranker-large，并保存真实排序与分数。上面的教学排序器仍是轻量启发式基线，两者结果分开报告。

```python
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

rerank_model = HuggingFaceCrossEncoder(model_name='BAAI/bge-reranker-large')
reranker = CrossEncoderReranker(model=rerank_model, top_n=3)
rerank_retriever = ContextualCompressionRetriever(
    base_compressor=reranker, base_retriever=retriever
)
reranked_docs = rerank_retriever.invoke(query)
```

In [3]:
case = cases["svm_soft_margin_alpha_bounds"]
chunks = load_default_chunks()
first_stage_search = build_bm25_chunk_search(chunks)
candidate_count = 6
retrieval_calls_before = 1
retrieval_calls_after = 1
dense_results = first_stage_search(case["query"], top_k=candidate_count)


def query_terms(text):
    return set(eu.TOKEN_RE.findall(eu.normalize_text(text).lower()))


def rerank_by_query_overlap(query, results):
    query_tokens = query_terms(query)
    scored = [(index, len(query_tokens.intersection(query_terms(item.text)))) for index, item in enumerate(results)]
    order = sorted(scored, key=lambda pair: (-pair[1], pair[0]))
    return [results[index] for index, _ in order]


reranked = rerank_by_query_overlap(case["query"], dense_results)
annotation = load_annotation(case['id'])


def rank_of_expected(results, expected_pages):
    expected = set(expected_pages)
    return next((rank for rank, item in enumerate(results, start=1) if expected.intersection(item.pages)), None)


before_rank = rank_of_expected(dense_results, annotation["expected_pages"])
after_rank = rank_of_expected(reranked, annotation["expected_pages"])
required_points = (("0⩽αi",), ("αi⩽C",))

def result_point_coverage(results):
    text = "".join("".join(item.text.split()) for item in results[:1]).replace("≤", "⩽")
    checks = [all(part in text for part in point) for point in required_points]
    return sum(checks) / len(checks), checks

before_points, before_checks = result_point_coverage(dense_results)
after_points, after_checks = result_point_coverage(reranked)
before_signature = [(item.chunk_id, tuple(item.pages), item.text) for item in dense_results]
after_signature = [(item.chunk_id, tuple(item.pages), item.text) for item in reranked]
same_candidates = sorted(before_signature) == sorted(after_signature)
before_chars = sum(len(item.text) for item in dense_results)
after_chars = sum(len(item.text) for item in reranked)
print("用户问题：", case["query"])
print("第一阶段候选页码：", [item.pages[0] for item in dense_results])
print("重排后候选页码：", [item.pages[0] for item in reranked])
print("正确资料排名：", before_rank, "→", after_rank)
print("第一条资料的回答要点覆盖率：", before_points, "→", after_points, "；检查：", before_checks, "→", after_checks)
print("候选数量：", len(dense_results), "→", len(reranked), "；候选字符数：", before_chars, "→", after_chars)
print("候选集合和文本完全相同：", same_candidates)
print("检索次数：", retrieval_calls_before, "→", retrieval_calls_after, "（重排在本地完成）")

assert len(dense_results) == len(reranked) == candidate_count and same_candidates and before_chars == after_chars
assert before_points == 0.0 and after_points == 1.0 and after_rank == 1
c5_svm_case = case
c5_svm_dense_results = list(dense_results)
c5_svm_candidate_count = candidate_count
c5_svm_before_rank = before_rank
c5_svm_before_points = before_points
emit_standard('重新排序候选资料', 'main', case['id'], dense_results, reranked, annotation['expected_pages'])

用户问题： 软间隔支持向量机对拉格朗日乘子 αi 有什么约束？
第一阶段候选页码： [130, 134, 62, 68, 67, 136]
重排后候选页码： [67, 134, 62, 130, 68, 136]
正确资料排名： 5 → 1
第一条资料的回答要点覆盖率： 0.0 → 1.0 ；检查： [False, False] → [True, True]
候选数量： 6 → 6 ；候选字符数： 1535 → 1535
候选集合和文本完全相同： True
检索次数： 1 → 1 （重排在本地完成）


## 压缩：有限长度内保留两项必要结论

牛顿法问题需要第 39 页中的两项原文要点：计算 Hessian 逆矩阵的代价，以及不能保证全局最优。第 39 页清理后有 1045 个字符；如果直接截取前 500 个字符，只能保留前一项。

下面根据问题中的两个主题词，在第 39 页各取一小段相邻文字，再放入同样的 500 字限制。主题词直接来自用户问题，不使用问题集里的预期页码或参考答案来检索。检查时要求两条完整说明都存在；压缩只减少送入回答的文字，不改变第一次检索。

信息压缩有两种位置。上下文压缩（Contextual Compression）在 retriever 返回文档后，按当前问题抽取相关句子或过滤整篇文档；它适合候选较多但每篇只有少量句子有用的情况。提示压缩（Prompt Compression）在上下文和指令已经组成 prompt 后，删除重复、低信息量 token；它适合 prompt 很长、且模型上下文窗口或 API 成本成为瓶颈的情况。两者都必须保留回答所需的原文依据，不能只用字符数下降来判断成功。

上下文压缩可以用 LangChain 的 `ContextualCompressionRetriever` 和 `EmbeddingsFilter`：相似度阈值太高会误删，太低则几乎没有压缩；阈值应在一批实际问题上比较，不能随意指定。LLMLingua 则使用一个较小的模型判断哪些 token 可以删去；它需要下载模型和额外算力，压缩后的 Prompt 仍要回到原文核对。

下面只展示两个外部压缩器的代码写法，不改变本页固定对照输出；实际接入前要重新核对原文证据是否被误删。

```python
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import EmbeddingsFilter

compressor = EmbeddingsFilter(embeddings=embedding, similarity_threshold=0.6)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=retriever
)
compressed_docs = compression_retriever.invoke(query)
```

```python
from llmlingua import PromptCompressor

compressor = PromptCompressor(model_name='microsoft/llmlingua-2-xlm-roberta-large-meetingbank')
compressed = compressor.compress_prompt(
    context, instruction='', question=question, rate=0.7,
    force_tokens=['\n', '。', '，', '：'],
)
final_prompt = compressed['compressed_prompt'] + '\n\n基于以上信息回答：' + question
```

In [4]:
case = cases["newton_hessian_cost"]
retrieval_calls_before = 1
retrieval_calls_after = 1
candidate_results = build_bm25_search(pages)("Hessian", top_k=2)
source_result = next(item for item in candidate_results if "Hessian" in item.text and "全局最优解" in item.text)
annotation = load_annotation(case['id'])
page_text = source_result.text
limit = 500


def take_around(text, anchors, before=120, after=220):
    spans = []
    for anchor in anchors:
        position = text.find(anchor)
        if position >= 0:
            spans.append((max(0, position - before), min(len(text), position + after)))
    spans.sort()
    merged = []
    for start, end in spans:
        if merged and start <= merged[-1][1]:
            merged[-1] = (merged[-1][0], max(merged[-1][1], end))
        else:
            merged.append((start, end))
    return clean_text(" … ".join(text[start:end] for start, end in merged))


page_text = clean_text(page_text)
plain_context = clean_text(candidate_results[0].text)[:limit]
compressed_context = take_around(page_text, ["Hessian", "全局最优"])[:limit]


required_claims = [
    ("Hessian矩阵的逆矩阵", "计算量通常较大"),
    ("并不一定保证最终求得的是全局最优解", "仅能保证其接近全局最优解"),
]


def claim_coverage(text):
    compact = compact_text(text)
    checks = [all(part in compact for part in claim) for claim in required_claims]
    return sum(checks) / len(checks), checks


before_coverage, before_checks = claim_coverage(plain_context)
after_coverage, after_checks = claim_coverage(compressed_context)
print("检索到的页面：", source_result.page)
print("页面字符数：", len(page_text), "；送入回答的上限：", limit)
print("检索次数：", retrieval_calls_before, "→", retrieval_calls_after, "；检索到的资料量：1 → 1")
print("实际送入回答的资料字符数：", len(plain_context), "→", len(compressed_context))
print("直接截取的两项检查：", before_checks)
print("删去无关文字后的两项检查：", after_checks)
print("完整结论覆盖率：", before_coverage, "→", after_coverage)
print("压缩后两项要点是否保留：", after_checks)

assert source_result.page == 39 and len(compressed_context) <= limit and before_coverage < after_coverage == 1.0
emit_standard('删去上下文中的无关文字', 'main', case['id'], candidate_results[:1], [source_result], annotation['expected_pages'], comparison={'name': '回答要点覆盖率', 'before': before_coverage, 'after': after_coverage, 'higher_is_better': True})

检索到的页面： 39
页面字符数： 986 ；送入回答的上限： 500
检索次数： 1 → 1 ；检索到的资料量：1 → 1
实际送入回答的资料字符数： 500 → 500
直接截取的两项检查： [False, False]
删去无关文字后的两项检查： [True, True]
完整结论覆盖率： 0.0 → 1.0
压缩后两项要点是否保留： [True, True]


## 标注来源：比较标注前后

下面使用一段为了演示而写的回答，不调用回答模型。标注前只有两条结论；程序先从问题检索得到的候选资料中寻找每条结论的原文依据，再把实际匹配到的页码加到结论后面；找不到依据时拒绝标注。这样可以直接比较“没有来源”和“每条结论都有来源”的差别。代码还会检查错页、漏引和没有原文依据的结论。

### 为什么要引用来源

参考引用表示回答中的每个主要结论来自哪些检索片段。它让读者可以追溯和核对，降低模型脱离资料编造的风险，也方便把页码换成标题、链接或文档 ID 做后处理。多个片段共同支持一个结论时，应列出全部必要来源；如果候选原文找不到支持，就拒绝标注或说明资料不足，不能为了让回答看起来完整而随便贴一个页码。

引用有两种常见实现。最简单的是在 prompt 中要求模型在陈述后写 `[来源1]`，并把带编号的文档交给模型；这种方式接入快，但需要后处理检查编号是否真的对应原文。更稳妥的是用 Pydantic 定义结构化输出，让模型返回回答、来源编号、原文片段和置信度，再由程序校验来源编号、原文是否存在以及置信度是否达到人工审核阈值。结构化输出要求模型支持对应格式，不能保证模型永远遵循 schema。

下面只展示结构化输出的代码写法，需要回答模型支持固定字段。本页结果仍是人工编写的示例和原文匹配检查，并不是模型生成或可信度评分。

```python
from typing import List
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser

class Citation(BaseModel):
    source_id: int = Field(description='来源编号')
    quote: str = Field(description='引用的原文片段')
class AnswerWithCitations(BaseModel):
    answer: str = Field(description='回答内容')
    citations: List[Citation] = Field(description='引用列表')
    confidence: float = Field(description='置信度，范围 0 到 1')

parser = PydanticOutputParser(pydantic_object=AnswerWithCitations)
structured_prompt = ChatPromptTemplate.from_template(
    '根据背景回答问题并提供引用。\n背景：{context}\n问题：{question}\n{format_instructions}'
)
chain = structured_prompt | llm | parser
result = chain.invoke({'context': context, 'question': question,
                       'format_instructions': parser.get_format_instructions()})
```

In [5]:
def citation_is_correct(claim, cited_page):
    if not isinstance(cited_page, int):
        return False
    cited_text = next((page["text"] for page in pages if page["page"] == cited_page), "")
    return compact_text(claim["evidence"]) in compact_text(cited_text)


case = cases["cross_validation_reliability"]
candidate_results = build_bm25_search(pages)("留出法", top_k=4)
annotation = load_annotation(case['id'])
claims = [
    {"conclusion": "交叉验证法是多次留出法，并更换测试子集。", "evidence": "交叉验证法本质上是在进行多次留出法，且每次都换不同的子集做测试集"},
    {"conclusion": "单次留出法只有一组训练集和测试集，偶然性较强。", "evidence": "一般的留出法只会划分出1组训练集和测试集，仅依靠1组训练集和测试集去对比不同算法之间的效果显然不够置信，偶然性太强"},
]

def find_support_page(claim, candidates):
    evidence = compact_text(claim["evidence"])
    return next(
        (item.page for item in candidates if evidence and evidence in compact_text(item.text)),
        None,
    )


def resolve_claim_pages(claims, candidates):
    resolved = []
    for claim in claims:
        page = find_support_page(claim, candidates)
        if page is None:
            raise ValueError(f"候选资料没有找到这条结论的原文依据：{claim['conclusion']}")
        resolved.append({**claim, "page": page})
    return resolved


resolved_claims = resolve_claim_pages(claims, candidate_results)
citation_after_results = [next(item for item in candidate_results if item.page == claim["page"]) for claim in resolved_claims]


def add_citations(claims):
    lines = []
    for claim in claims:
        if not citation_is_correct(claim, claim["page"]):
            raise ValueError(f"这条结论没有找到对应原文：{claim['conclusion']}")
        lines.append(f"- {claim['conclusion']}（《南瓜书》第 {claim['page']} 页）")
    return "\n".join(lines)


def citation_coverage(answer, claims):
    checks = [
        claim["conclusion"] in answer
        and f"第 {claim['page']} 页" in answer
        and citation_is_correct(claim, claim["page"])
        for claim in claims
    ]
    return sum(checks) / len(checks), checks


plain_answer = "\n".join(f"- {claim['conclusion']}" for claim in claims)
cited_answer = add_citations(resolved_claims)
before_coverage, before_checks = citation_coverage(plain_answer, resolved_claims)
after_coverage, after_checks = citation_coverage(cited_answer, resolved_claims)
supported_checks = [citation_is_correct(claim, claim["page"]) for claim in resolved_claims]
wrong_page_check = not citation_is_correct(resolved_claims[0], 18)
missing_citation_check = not citation_is_correct(resolved_claims[0], None)
unsupported_claim = {"conclusion": "书中没有这项结论", "evidence": "书中没有这项结论"}
try:
    resolve_claim_pages([unsupported_claim], candidate_results)
except ValueError as error:
    unsupported_check = True
    unsupported_error = str(error)
else:
    unsupported_check = False
    unsupported_error = "没有拒绝无依据结论"
check_case = cases["gaussian_mean_estimate"]
check_candidates = build_bm25_search(pages)(check_case["query"], top_k=4)
check_annotation = load_annotation(check_case['id'])
check_claim = {"conclusion": "高斯分布的均值参数可以用样本均值估计。", "evidence": "ˆµc = ¯x = 1 n n X i=1 xi"}
resolved_check_claim = resolve_claim_pages([check_claim], check_candidates)[0]
check_supported = citation_is_correct(resolved_check_claim, resolved_check_claim["page"])
check_plain = f"- {check_claim['conclusion']}"
check_cited = add_citations([resolved_check_claim])
check_before, _ = citation_coverage(check_plain, [resolved_check_claim])
check_after, _ = citation_coverage(check_cited, [resolved_check_claim])
print("标注前（示例文字，不是模型输出）：")
print(plain_answer)
print("标注后：")
print(cited_answer)
print("用于标注的候选页面：", [item.page for item in candidate_results])
print("从候选原文匹配得到的页码：", [claim["page"] for claim in resolved_claims])
print("有来源的结论比例：", before_coverage, "→", after_coverage, "；检查：", before_checks, "→", after_checks)
print("逐条结论有对应原文：", supported_checks)
print("错页、漏引、无依据结论检查：", [wrong_page_check, missing_citation_check, unsupported_check])
print("无依据结论拒绝标注：", unsupported_error)
print("换题检查：高斯均值参数的候选页面", [item.page for item in check_candidates], "；证据匹配页", resolved_check_claim["page"], "；有来源的结论比例：", check_before, "→", check_after)

assert candidate_results[0].page == 162 and resolved_claims[0]["page"] == 19 and before_coverage == 0.0 and after_coverage == 1.0
assert all(supported_checks) and wrong_page_check and missing_citation_check and unsupported_check
assert check_candidates[0].page == resolved_check_claim["page"] == 77 and check_supported and check_before == 0.0 and check_after == 1.0
emit_standard('回答标注来源', 'main', case['id'], candidate_results, citation_after_results, annotation['expected_pages'], '再次改善', comparison={'name': '有来源结论比例', 'before': before_coverage, 'after': after_coverage, 'higher_is_better': True})
emit_standard('回答标注来源', 'check', check_case['id'], check_candidates, [check_candidates[0]], check_annotation['expected_pages'], '再次改善', comparison={'name': '有来源结论比例', 'before': check_before, 'after': check_after, 'higher_is_better': True})

标注前（示例文字，不是模型输出）：
- 交叉验证法是多次留出法，并更换测试子集。
- 单次留出法只有一组训练集和测试集，偶然性较强。
标注后：
- 交叉验证法是多次留出法，并更换测试子集。（《南瓜书》第 19 页）
- 单次留出法只有一组训练集和测试集，偶然性较强。（《南瓜书》第 19 页）
用于标注的候选页面： [162, 19, 18, 138]
从候选原文匹配得到的页码： [19, 19]
有来源的结论比例： 0.0 → 1.0 ；检查： [False, False] → [True, True]
逐条结论有对应原文： [True, True]
错页、漏引、无依据结论检查： [True, True, True]
无依据结论拒绝标注： 候选资料没有找到这条结论的原文依据：书中没有这项结论
换题检查：高斯均值参数的候选页面 [77, 184, 80, 78] ；证据匹配页 77 ；有来源的结论比例： 0.0 → 1.0


## 资料不足就说明边界

用户询问书中建议的 CUDA 版本，但整本书没有出现 CUDA。普通流程仍会把最相近的一页交给回答，内容却答非所问。加入资料检查后，程序会从问题中提取大写技术名称，并确认它是否在整本书中出现；找不到时不把无关页面交给回答，而是说明资料不足。同时用资料内的正常问题复核，避免拒答规则误伤可回答的问题。此处按 ASCII 字母和数字识别全大写技术名称，中文与 CUDA/MNIST 之间没有空格也能匹配，并按完整标识核对原文。这个简单规则只适合名称明确的问题；名称出现只能排除“名称不存在”，不能证明原文足以回答，更一般的问题仍需要单独判断资料是否足够。

In [6]:
case = cases["book_evidence_boundary"]
boundary_controls = load_query_controls(case["id"])
supported_query = boundary_controls["steps"][0]["query"]
unsupported_query = boundary_controls["steps"][1]["query"]
page_search = build_bm25_search(pages)

def extract_from_source(query, counter):
    counter["calls"] += 1
    source_result = page_search(query, top_k=1)[0]
    source_body = clean_text(re.sub(r"→_→.*?←_←", "", source_result.text))
    source_sentences = [
        sentence.strip()
        for sentence in re.split(r"(?<=[。！？])\s+", source_body)
        if sentence.strip()
    ]
    rows = [{"page": index, "text": text} for index, text in enumerate(source_sentences)]
    result = build_bm25_search(rows)(query, top_k=1)[0]
    extracted = source_sentences[result.page]
    marker = "具体来说，"
    answer = extracted.split(marker, 1)[1] if marker in extracted else extracted
    return source_result.page, answer


def missing_technical_identifiers(query):
    # 中文字符也是 Unicode 的“词字符”，不能用 \\b 判断英文标识的边界。
    pattern = r"(?<![A-Za-z0-9_])[A-Z][A-Z0-9]*(?:[./-][A-Z0-9]+)*(?:\+{1,2})?(?![A-Za-z0-9_])"
    identifiers = dict.fromkeys(re.findall(pattern, query))
    corpus_identifiers = {
        identifier for page in pages for identifier in re.findall(pattern, page["text"])
    }
    return [identifier for identifier in identifiers if identifier not in corpus_identifiers]


def answer_from_source(query, check_boundary=False, counter=None):
    counter = counter if counter is not None else {"calls": 0}
    page, extracted = extract_from_source(query, counter)
    missing = missing_technical_identifiers(query) if check_boundary else []
    if missing:
        return page, f"资料中没有找到回答“{query}”所需的 {'、'.join(missing)} 相关说明，不能仅依据这份资料回答。"
    return page, extracted


baseline_counter = {"calls": 0}
checked_counter = {"calls": 0}
baseline_results = [
    answer_from_source(supported_query, counter=baseline_counter),
    answer_from_source(unsupported_query, counter=baseline_counter),
]
checked_answers = [
    answer_from_source(supported_query, check_boundary=True, counter=checked_counter),
    answer_from_source(unsupported_query, check_boundary=True, counter=checked_counter),
]
annotation = load_annotation(case['id'])
baseline_answers = [answer for _, answer in baseline_results]
checked_texts = [answer for _, answer in checked_answers]
baseline_pass = ["业务场景" in baseline_answers[0], "资料中没有找到" in baseline_answers[1]]
checked_pass = ["业务场景" in checked_texts[0], unsupported_query in checked_texts[1] and "CUDA" in checked_texts[1]]
normal_points, normal_checks = answer_point_coverage(checked_texts[0], (("模型评估与选择",), ("业务场景",)))
baseline_unsupported_points, _ = answer_point_coverage(baseline_answers[1], (("资料中没有找到",),))
checked_unsupported_points, unsupported_checks = answer_point_coverage(checked_texts[1], (("资料中没有找到",), ("不能仅依据",)))

print("正常问题：", supported_query)
print("检索到的页面：", checked_answers[0][0])
print("检查后的回答（截取）：", preview(checked_texts[0]))
print("正常问题回答要点覆盖率：", normal_points, "；检查：", normal_checks)
print("资料中缺少的大写技术名称：", missing_technical_identifiers(unsupported_query))
print("资料外问题：", unsupported_query)
print("检查前（截取）：", preview(baseline_answers[1]))
print("检查后：", checked_texts[1])
print("资料外问题拒答要点覆盖率：", baseline_unsupported_points, "→", checked_unsupported_points, "；检查：", unsupported_checks)
print("两项要求是否通过：", baseline_pass, "→", checked_pass)
print("送入回答的资料量：", 2, "→", 1, "；检索次数：", baseline_counter["calls"], "→", checked_counter["calls"])

cuda_adjacent_query = unsupported_query.replace(" CUDA ", "CUDA")
_, cuda_adjacent_answer = answer_from_source(cuda_adjacent_query, check_boundary=True)
print("中文紧邻标识复查：", cuda_adjacent_query, "→", cuda_adjacent_answer)
assert missing_technical_identifiers(cuda_adjacent_query) == ["CUDA"]
assert "资料中没有找到" in cuda_adjacent_answer and "不能仅依据" in cuda_adjacent_answer

assert baseline_results[0] == checked_answers[0] and normal_points == 1.0
assert missing_technical_identifiers(unsupported_query) == ["CUDA"] and checked_pass == [True, True] and checked_unsupported_points == 1.0
guardrail_before = {'pages': [int(baseline_results[1][0])], 'first_required_rank': 1, 'required_page_coverage': 0.0}
guardrail_after = {'pages': [], 'first_required_rank': None, 'required_page_coverage': 0.0}
emit_standard('资料不足时拒答', 'main', case['id'], [], [], annotation['expected_pages'], '再次改善', guardrail_before, guardrail_after, comparison={'name': '拒答要点覆盖率', 'before': baseline_unsupported_points, 'after': checked_unsupported_points, 'higher_is_better': True})

正常问题： 南瓜书第 2 章把模型评估与选择描述为什么目标？
检索到的页面： 18
检查后的回答（截取）： 介绍内容正如本章名称 “模型评估与选择”所述，讲述的是如何评估模型的优劣和选择最适合自己业务场景的模型。
正常问题回答要点覆盖率： 1.0 ；检查： [True, True]


资料中缺少的大写技术名称： ['CUDA']
资料外问题： 《南瓜书》建议使用哪个 CUDA 版本训练模型？
检查前（截取）： 使用说明 • 南瓜书的所有内容都是以西瓜书的内容为前置知识进行表述的，所以南瓜书的最佳使用方法是以西瓜书 为主线，遇到自己推导不出来或者看不懂的公式时再来查阅南瓜书； • 对于初学机器学习的小白，西瓜书第1 章和第2 章的公式强烈不建议深究，简单过一下即可，等你学得 有点飘的时候再回来啃都来得及； • 每个公式的解析和推导我们都力(zhi) 争(neng) ...
检查后： 资料中没有找到回答“《南瓜书》建议使用哪个 CUDA 版本训练模型？”所需的 CUDA 相关说明，不能仅依据这份资料回答。
资料外问题拒答要点覆盖率： 0.0 → 1.0 ；检查： [True, True]
两项要求是否通过： [True, False] → [True, True]
送入回答的资料量： 2 → 1 ；检索次数： 2 → 2
中文紧邻标识复查： 《南瓜书》建议使用哪个CUDA版本训练模型？ → 资料中没有找到回答“《南瓜书》建议使用哪个CUDA版本训练模型？”所需的 CUDA 相关说明，不能仅依据这份资料回答。


## 二次检查：Fβ 已经排在第一条时不应被改坏

这道 Fβ 问题也直接使用完整用户问题做第一阶段检索；第 20 页已经排在第一条，所以它只作为不改坏的复查，不把它当成重排有效的正例。重排仍只处理原有候选，预期信息只用于事后检查。

In [7]:
case = cases["f1_harmonic_mean"]
dense_results = first_stage_search(case["query"], top_k=6)
annotation = load_annotation(case['id'])
reranked = rerank_by_query_overlap(case["query"], dense_results)

def f1_rank(results, expected_pages):
    expected = set(expected_pages)
    return next((i for i, item in enumerate(results, start=1) if expected.intersection(item.pages)), None)

def f1_point_coverage(results):
    text = compact_text(results[0].text)
    checks = ["加权调和平均" in text and "算数平均" in text, "更重视较小值" in text]
    return sum(checks) / len(checks), checks

before_points, before_checks = f1_point_coverage(dense_results)
after_points, after_checks = f1_point_coverage(reranked)
before_signature = [(item.chunk_id, tuple(item.pages), item.text) for item in dense_results]
after_signature = [(item.chunk_id, tuple(item.pages), item.text) for item in reranked]
same_candidates = sorted(before_signature) == sorted(after_signature)
before_chars = sum(len(item.text) for item in dense_results)
after_chars = sum(len(item.text) for item in reranked)
print("用户问题：", case["query"])
print("第一阶段候选页码：", [item.pages[0] for item in dense_results])
print("重排后候选页码：", [item.pages[0] for item in reranked])
print("正确资料排名：", f1_rank(dense_results, annotation["expected_pages"]), "→", f1_rank(reranked, annotation["expected_pages"]))
print("第一条资料的回答要点覆盖率：", before_points, "→", after_points, "；检查：", before_checks, "→", after_checks)
print("候选数量：", len(dense_results), "→", len(reranked), "；候选字符数：", before_chars, "→", after_chars)
print("候选集合和文本完全相同：", same_candidates, "；检索次数：1 → 1")
print("复查结论：原本排在第一条的 Fβ 资料没有被重排破坏。")

assert len(dense_results) == len(reranked) == 6 and same_candidates and before_chars == after_chars and before_points == after_points == 1.0 and f1_rank(dense_results, annotation["expected_pages"]) == f1_rank(reranked, annotation["expected_pages"]) == 1
emit_standard('重新排序候选资料', 'check', case['id'], dense_results, reranked, annotation['expected_pages'], '确认没有改坏')

用户问题： 为什么 Fβ 比算术平均更强调较小的指标？
第一阶段候选页码： [20, 20, 151, 141, 95, 26]
重排后候选页码： [20, 20, 141, 151, 95, 26]
正确资料排名： 1 → 1
第一条资料的回答要点覆盖率： 1.0 → 1.0 ；检查： [True, True] → [True, True]
候选数量： 6 → 6 ；候选字符数： 1536 → 1536
候选集合和文本完全相同： True ；检索次数：1 → 1
复查结论：原本排在第一条的 Fβ 资料没有被重排破坏。


## 二次检查：压缩对数几率回归的说明

对数几率回归所在的第 36 页前半段是推导过程，直接截取前 500 个字符会漏掉阈值判定。压缩时只围绕用户问题已有的“阈值、正例、反例”保留相邻文字，使同一页中的两项结论都进入回答。

In [8]:
case = cases["logistic_regression_threshold"]
source_result = build_bm25_search(pages)(case["query"], top_k=1)[0]
annotation = load_annotation(case['id'])
page_text = source_result.text
limit = 500
plain_context = page_text[:limit]
compressed_context = take_around(page_text, ["阈值", "正例", "反例"])[:limit]

def logistic_point_coverage(text):
    compact = compact_text(text)
    checks = ["通常设为θ=0.5" in compact, "如果yi⩾θ则判xi为正例，反之判为反例" in compact]
    return sum(checks) / len(checks), checks

before_points, before_checks = logistic_point_coverage(plain_context)
after_points, after_checks = logistic_point_coverage(compressed_context)
print("检索问题：", case["query"], "；页面：", source_result.page)
print("直接截取的两项检查：", before_checks)
print("压缩后的两项检查：", after_checks)
print("回答要点覆盖率：", before_points, "→", after_points)
print("资料量：1 → 1；实际送入回答的字符数：", len(plain_context), "→", len(compressed_context), "；检索次数：1 → 1")
print("明确结论：压缩去掉了前置推导的无关文字，阈值和正反例判定两项均保留。")

assert source_result.page == 36 and before_points == 0.0 and after_points == 1.0
emit_standard('删去上下文中的无关文字', 'check', case['id'], [source_result], [source_result], annotation['expected_pages'], '再次改善', comparison={'name': '回答要点覆盖率', 'before': before_points, 'after': after_points, 'higher_is_better': True})

检索问题：

 对数几率回归概率阈值如何判为正例或反例？ ；页面： 36
直接截取的两项检查： [False, False]
压缩后的两项检查： [True, True]
回答要点覆盖率： 0.0 → 1.0
资料量：1 → 1；实际送入回答的字符数： 500 → 378 ；检索次数：1 → 1
明确结论：压缩去掉了前置推导的无关文字，阈值和正反例判定两项均保留。


## 二次检查：书中没有 MNIST 资料时拒答

这个问题询问书中是否给出 MNIST 训练样本数。全文检索会返回相近但无关的页面，不能把它当作答案。先记录直接检索的结果，再检查 MNIST 是否真的出现在书中；找不到就停止回答，并移除无关页面。前面的 CUDA 示例也确认了资料内的正常问题仍能回答。

In [9]:
case = cases["book_evidence_mnist"]
guardrail_controls = load_query_controls(case["id"])
query = guardrail_controls["steps"][0]["query"]
baseline_counter = {"calls": 0}
checked_counter = {"calls": 0}
baseline_page, baseline_answer = answer_from_source(query, counter=baseline_counter)
checked_page, checked_answer = answer_from_source(query, check_boundary=True, counter=checked_counter)
missing = missing_technical_identifiers(query)
annotation = load_annotation(case['id'])

def refusal_point_coverage(text):
    checks = ["资料中没有找到" in text, "不能仅依据" in text, "MNIST" in text]
    return sum(checks) / len(checks), checks

before_points, before_checks = refusal_point_coverage(baseline_answer)
after_points, after_checks = refusal_point_coverage(checked_answer)
print("问题：", query, "；检索页：", baseline_page, "→", checked_page)
print("检查前（截取）：", preview(baseline_answer))
print("检查后：", checked_answer)
print("缺少的名称：", missing)
print("拒答要点覆盖率：", before_points, "→", after_points, "；检查：", before_checks, "→", after_checks)
print("送入回答的资料量：1 → 0；检索次数：", baseline_counter["calls"], "→", checked_counter["calls"])
print("资料内正常问题检查：", normal_points, "；明确结论：没有 MNIST 证据，不用相近页面猜答案。")

mnist_adjacent_query = query.replace(" MNIST ", "MNIST")
_, mnist_adjacent_answer = answer_from_source(mnist_adjacent_query, check_boundary=True)
print("中文紧邻标识复查：", mnist_adjacent_query, "→", mnist_adjacent_answer)
assert missing_technical_identifiers(mnist_adjacent_query) == ["MNIST"]
assert "资料中没有找到" in mnist_adjacent_answer and "不能仅依据" in mnist_adjacent_answer

assert missing == ["MNIST"] and baseline_page == checked_page
assert before_points == 0.0 and after_points == 1.0 and normal_points == 1.0
guardrail_before = {'pages': [int(baseline_page)], 'first_required_rank': 1, 'required_page_coverage': 0.0}
guardrail_after = {'pages': [], 'first_required_rank': None, 'required_page_coverage': 0.0}
emit_standard('资料不足时拒答', 'check', case['id'], [], [], annotation['expected_pages'], '再次改善', guardrail_before, guardrail_after, comparison={'name': '拒答要点覆盖率', 'before': before_points, 'after': after_points, 'higher_is_better': True})

问题： 《南瓜书》给出了 MNIST 数据集的训练样本数吗？ ；检索页： 2 → 2
检查前（截取）： 前言 “周志华老师的《机器学习》（西瓜书）是机器学习领域的经典入门教材之一，周老师为了使尽可能多的读 者通过西瓜书对机器学习有所了解, 所以在书中对部分公式的推导细节没有详述，但是这对那些想深究公式推 导细节的读者来说可能“不太友好”，本书旨在对西瓜书里比较难理解的公式加以解析，以及对部分公式补充 具体的推导细节。” 读到这里，大家可能会疑问为啥前面这段话加...
检查后： 资料中没有找到回答“《南瓜书》给出了 MNIST 数据集的训练样本数吗？”所需的 MNIST 相关说明，不能仅依据这份资料回答。
缺少的名称： ['MNIST']
拒答要点覆盖率： 0.0 → 1.0 ；检查： [False, False, False] → [True, True, True]
送入回答的资料量：1 → 0；检索次数： 1 → 1
资料内正常问题检查： 1.0 ；明确结论：没有 MNIST 证据，不用相近页面猜答案。
中文紧邻标识复查： 《南瓜书》给出了MNIST数据集的训练样本数吗？ → 资料中没有找到回答“《南瓜书》给出了MNIST数据集的训练样本数吗？”所需的 MNIST 相关说明，不能仅依据这份资料回答。


## 使用顺序

先确认正确资料是否已经找到：位置太后就重新排序，内容太长再删去无关文字。生成答案时给主要结论标注来源；资料中找不到问题所需的内容，就明确说明资料不足。

重新排序、压缩、来源标注和资料是否足够的检查都保留了原问题的必要要点。重新排序部分把软间隔支持向量机的约束片段从第 5 条提到第 1 条；Fβ 复查确认原本排在第一条的资料没有被改坏；对数几率回归在同一页内保留阈值与正反例判定；MNIST 因全文无证据而停止猜测。来源标注说明了同一段示例文字从没有页码到逐条附上已核对页码的变化，但不代表模型回答已经生成或评测。

实际选型可以按这个顺序：先确认正确片段是否已经召回；已召回但排得靠后时重排；片段太长时做上下文或 prompt 压缩；回答中逐条附上能在候选原文中找到的来源；资料根本没有所需内容时明确拒答。重排和压缩可以组合，但应分别记录候选集合、字符数、必要要点覆盖率、延迟和成本，不能只看最终回答听起来是否流畅。相似度阈值、保留数量和压缩率都要在开发集上调节，过激会漏掉证据，过松则无法降低噪声。

## 可选真实实验：BAAI/bge-reranker-large 同题比较

这一段是可选的真实本地实验，不替代上面的轻量启发式教学例子。它复用本 Notebook 已从 canonical 数据包读取的 SVM 问题、同一个第一阶段 BM25 候选列表和相同的 top-k=6；唯一改变是用真实 BAAI/bge-reranker-large Cross-Encoder 对这 6 条候选逐条打分，再保留同样的 6 条排序结果。评价只比较 canonical 标注的必要页是否进入第 1 条，不能把一次运行外推为所有数据集都会改善。

在 fresh clone 中先在 llm-universe-c7 kernel 对应的环境，从 C7 唯一依赖入口准备依赖：

    python -m pip install -r "notebook/C7 高级 RAG 技巧/requirements-c7.txt"

依赖安装完成后，再单独准备本实验需要的模型缓存：

    python -c "from modelscope import snapshot_download; print(snapshot_download('BAAI/bge-reranker-large'))"

第二条命令只下载并缓存指定的 BAAI/bge-reranker-large；Notebook 运行时使用 local_files_only=True，不会切换模型或静默联网。缓存、依赖或配置不完整时直接报错。真实输出会保存模型 ID、canonical query/evidence、6 条候选的前后顺序、Cross-Encoder 分数、top-1 指标和本次 outcome（改善/不变/退化），并明确本例的观察范围。

In [10]:
import time
from modelscope import snapshot_download
from langchain_core.documents import Document
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from common.dataset import load_evidence_records, load_query_records

RERANKER_MODEL_ID = "BAAI/bge-reranker-large"
case = c5_svm_case
dense_results = c5_svm_dense_results
candidate_count = c5_svm_candidate_count
before_rank = c5_svm_before_rank
before_points = c5_svm_before_points
RERANKER_TOP_K = candidate_count
RERANKER_CONTEXT_TOP_K = 1

canonical_query = case["query"]
assert len(dense_results) == RERANKER_TOP_K == candidate_count

try:
    reranker_model_dir = Path(
        snapshot_download(RERANKER_MODEL_ID, local_files_only=True)
    )
except Exception as error:
    raise RuntimeError(
        "本实验需要已缓存的 BAAI/bge-reranker-large；请先执行本单元上方的模型准备命令。"
    ) from error

required_model_files = ("config.json", "tokenizer.json")
weight_files = ("pytorch_model.bin", "model.safetensors")
missing_model_files = [
    filename for filename in required_model_files
    if not (reranker_model_dir / filename).is_file()
]
if missing_model_files or not any(
    (reranker_model_dir / filename).is_file() for filename in weight_files
):
    raise FileNotFoundError(
        f"本地 Cross-Encoder 缓存不完整：{reranker_model_dir}；"
        f"缺少 {missing_model_files or '模型权重'}"
    )

reranker_model = HuggingFaceCrossEncoder(
    model_name=str(reranker_model_dir),
    model_kwargs={"device": "cpu", "max_length": 512},
)
cross_encoder_reranker = CrossEncoderReranker(
    model=reranker_model,
    top_n=RERANKER_TOP_K,
)
cross_documents = [
    Document(
        page_content=item.text,
        metadata={"chunk_id": item.chunk_id, "pages": list(item.pages)},
    )
    for item in dense_results
]
pairs = [(canonical_query, item.text) for item in dense_results]
score_started = time.perf_counter()
raw_scores = [float(score) for score in reranker_model.score(pairs)]
score_seconds = time.perf_counter() - score_started
cross_documents_ranked = list(
    cross_encoder_reranker.compress_documents(cross_documents, canonical_query)
)
cross_encoder_ranked_ids = [
    document.metadata["chunk_id"] for document in cross_documents_ranked
]
scores_by_chunk = {
    item.chunk_id: score for item, score in zip(dense_results, raw_scores)
}
score_order = [
    item.chunk_id
    for item in sorted(
        dense_results,
        key=lambda item: (-scores_by_chunk[item.chunk_id], dense_results.index(item)),
    )
]
cross_encoder_ranked = [
    next(item for item in dense_results if item.chunk_id == chunk_id)
    for chunk_id in cross_encoder_ranked_ids
]

# Only after Cross-Encoder scoring and reranking may evaluation labels be read.
canonical_query_record = next(
    row for row in load_query_records() if row["query_id"] == case["id"]
)
canonical_evidence_ids = sorted({
    evidence_id
    for claim in canonical_query_record["reference_claims"]
    for evidence_id in claim["evidence_ids"]
})
canonical_evidence = {
    row["evidence_id"]: row
    for row in load_evidence_records()
    if row["evidence_id"] in canonical_evidence_ids
}
annotation = load_annotation(case["id"])

assert canonical_query == case["query"]
assert canonical_evidence_ids and set(canonical_evidence) == set(canonical_evidence_ids)
assert set(annotation["expected_pages"]) == {
    int(canonical_evidence[evidence_id]["page"])
    for evidence_id in canonical_evidence_ids
}
assert set(annotation["expected_pages"]).issubset({
    page for item in dense_results for page in item.pages
})
before_ids = [item.chunk_id for item in dense_results]
after_ids = [item.chunk_id for item in cross_encoder_ranked]
same_candidates = (
    len(after_ids) == len(before_ids) == RERANKER_TOP_K
    and set(before_ids) == set(after_ids)
    and score_order == after_ids
)
cross_before_rank = rank_of_expected(dense_results, annotation["expected_pages"])
cross_after_rank = rank_of_expected(cross_encoder_ranked, annotation["expected_pages"])
cross_before_points, cross_before_checks = result_point_coverage(dense_results)
cross_after_points, cross_after_checks = result_point_coverage(cross_encoder_ranked)

def rank_outcome(before, after):
    if before is None or after is None:
        return "无法比较"
    if after < before:
        return "改善"
    if after > before:
        return "退化"
    return "不变"

outcome = rank_outcome(cross_before_rank, cross_after_rank)
print("模型：", RERANKER_MODEL_ID)
print("canonical query：", canonical_query)
print("canonical evidence_id：", canonical_evidence_ids)
print("候选 top-k（相同）：", RERANKER_TOP_K)
print("候选 chunk_id：", before_ids)
print("Cross-Encoder 分数：", scores_by_chunk)
print("重排后 chunk_id：", after_ids)
print("必要页排名：", cross_before_rank, "→", cross_after_rank)
print("top-1 要点覆盖率：", cross_before_points, "→", cross_after_points)
print("候选集合完全相同：", same_candidates)
print("打分耗时（秒）：", round(score_seconds, 3))
print("本次 outcome：", outcome)
print("范围：只对一个 canonical query 的同一 6 条候选做一次本地比较，不外推到整个数据集。")

assert same_candidates
assert cross_before_rank == before_rank == 5
assert cross_after_rank == 1
assert cross_before_points == before_points == 0.0
assert cross_after_points == 1.0
assert cross_after_checks == [True, True]
assert outcome == "改善"

emit_tutorial_audit({
    "case_id": case["id"],
    "method": "可选真实 Cross-Encoder 同题比较",
    "role": "optional",
    "experiment": "optional_real_cross_encoder_same_query",
    "model": RERANKER_MODEL_ID,
    "model_id": RERANKER_MODEL_ID,
    "query_id": canonical_query_record["query_id"],
    "query": canonical_query,
    "canonical_evidence_ids": canonical_evidence_ids,
    "candidate_top_k": RERANKER_TOP_K,
    "answer_metric_top_k": RERANKER_CONTEXT_TOP_K,
    "candidate_chunk_ids_before": before_ids,
    "candidate_chunk_ids_after": after_ids,
    "scores": scores_by_chunk,
    "required_pages": annotation["expected_pages"],
    "rank_before": cross_before_rank,
    "rank_after": cross_after_rank,
    "top1_point_coverage_before": cross_before_points,
    "top1_point_coverage_after": cross_after_points,
    "outcome": outcome,
    "score_seconds": round(score_seconds, 3),
    "scope": "单个 canonical query、同一 6 条候选的一次本地运行；不外推整体效果。",
})


/usr/local/Caskroom/miniconda/base/envs/py310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-09-12 15:02:42,499 - modelscope - WARNING - We can not confirm the cached file is for revision: master


模型： BAAI/bge-reranker-large
canonical query： 软间隔支持向量机对拉格朗日乘子 αi 有什么约束？
canonical evidence_id： ['evi_df6d541be200']
候选 top-k（相同）： 6
候选 chunk_id： ['c679', 'c696', 'c357', 'c385', 'c381', 'c704']
Cross-Encoder 分数： {'c679': 0.11332753300666809, 'c696': 0.40311962366104126, 'c357': 0.3213668167591095, 'c385': 0.30329903960227966, 'c381': 0.9211874008178711, 'c704': 0.003013661364093423}
重排后 chunk_id： ['c381', 'c696', 'c357', 'c385', 'c679', 'c704']
必要页排名： 5 → 1
top-1 要点覆盖率： 0.0 → 1.0
候选集合完全相同： True
打分耗时（秒）： 6.04
本次 outcome： 改善
范围：只对一个 canonical query 的同一 6 条候选做一次本地比较，不外推到整个数据集。
